

# Evidencia de Aprendizaje 2: Manejo de archivos CSV, JSON y TXT

**Objetivo:** leer, modificar y guardar archivos CSV y JSON mediante clases de Python, y generar un informe TXT con las fuentes y los resultados.

Se conservan los archivos originales y las modificaciones se guardan en archivos separados.

In [4]:
from pathlib import Path
import sys
import csv
import json

# Agregar la carpeta raíz del proyecto a la ruta de Python
RAIZ_PROYECTO = Path.cwd()

if RAIZ_PROYECTO.name == "AE2":
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent

sys.path.insert(0, str(RAIZ_PROYECTO))

from AE2.manejador_csv import ManejadorCSV
from AE2.manejador_json import ManejadorJSON
from AE2.generador_resumen_txt import GeneradorResumenTXT

URL_CSV = "https://huggingface.co/datasets/zhugetd/HeightData"
URL_JSON = "https://huggingface.co/datasets/loge2146/evocorps-neutral-personas"

RUTA_CSV = RAIZ_PROYECTO / "AE2" / "datos_originales" / "heights.csv"
RUTA_JSON = RAIZ_PROYECTO / "AE2" / "datos_originales" / "neutral_personas_database.json"

SALIDA_CSV = RAIZ_PROYECTO / "AE2" / "datos_modificados" / "heights_modificado.csv"
SALIDA_JSON = RAIZ_PROYECTO / "AE2" / "datos_modificados" / "neutral_personas_database_modificado.json"

print("Raíz del proyecto:", RAIZ_PROYECTO)
print("CSV encontrado:", RUTA_CSV.exists())
print("JSON encontrado:", RUTA_JSON.exists())

Raíz del proyecto: /workspaces/AD_2026Q4_Grupo20
CSV encontrado: True
JSON encontrado: True


## 1. Carga y revisión de los archivos originales

Se leen los archivos locales utilizando las clases creadas. Primero se visualizan algunos registros antes de realizar modificaciones.

In [5]:
manejador_csv = ManejadorCSV(str(RUTA_CSV))
encabezados_csv, datos_csv = manejador_csv.cargar_datos()

manejador_json = ManejadorJSON(str(RUTA_JSON))
personas_json = manejador_json.cargar_datos()

print("Columnas CSV:", encabezados_csv)
print("Cantidad de registros CSV:", len(datos_csv))
print("Primeros 3 registros CSV:")
for fila in datos_csv[:3]:
    print(fila)

print("\nCantidad de registros JSON:", len(personas_json))
print("Primer registro JSON:")
print(json.dumps(personas_json[0], ensure_ascii=False, indent=2))

Columnas CSV: ['id', 'name', 'height']
Cantidad de registros CSV: 584
Primeros 3 registros CSV:
['animals-african_elephant', 'African Elephant', '4']
['animals-ant', 'Ant', '0.002']
['animals-asian_elephant', 'Asian Elephant', '3.5']

Cantidad de registros JSON: 200
Primer registro JSON:
{
  "id": "neutral_robert_001",
  "type": "neutral",
  "name": "Robert",
  "profession": "Delivery Person",
  "demographics": {
    "age": "35",
    "region": "Saint Peters, MO, USA"
  },
  "background": "A Delivery Person who watches how friends respond to news and adapts their view accordingly. Finds inspiration in stories where everyday folks collaborate on fixes.",
  "personality_traits": [
    "Switches tone based on how others respond",
    "Worries about missing context and second-guesses self",
    "Looks for human impact whenever policies trend"
  ],
  "communication_style": {
    "tone": "Thoughtful but impressionable",
    "engagement_level": "low"
  }
}


## 2. Modificación y almacenamiento

En el CSV se modifica el nombre del primer registro. En el JSON se agrega el campo `reviewed` al primer objeto. Las modificaciones se guardan en nuevos archivos, sin reemplazar los originales.

In [6]:
cambio_csv = manejador_csv.modificar_datos()
manejador_csv.guardar_datos(str(SALIDA_CSV))

cambio_json = manejador_json.modificar_datos()
manejador_json.guardar_datos(str(SALIDA_JSON))

print("Cambio realizado en CSV:", cambio_csv)
print("Cambio realizado en JSON:", cambio_json)
print("\nArchivos generados:")
print(SALIDA_CSV, "-", SALIDA_CSV.exists())
print(SALIDA_JSON, "-", SALIDA_JSON.exists())

Cambio realizado en CSV: ('African Elephant', 'African Elephant (revisado)')
Cambio realizado en JSON: (False, True)

Archivos generados:
/workspaces/AD_2026Q4_Grupo20/AE2/datos_modificados/heights_modificado.csv - True
/workspaces/AD_2026Q4_Grupo20/AE2/datos_modificados/neutral_personas_database_modificado.json - True


## 3 Verificación de los archivos modificados

Se vuelven a leer los archivos de salida para comprobar que los cambios quedaron guardados correctamente.

In [7]:
with SALIDA_CSV.open("r", encoding="utf-8-sig", newline="") as archivo:
    filas_csv_final = list(csv.reader(archivo))

with SALIDA_JSON.open("r", encoding="utf-8") as archivo:
    personas_json_final = json.load(archivo)

print("Primer registro CSV modificado:")
print(filas_csv_final[1])

print("\nPrimer registro JSON modificado:")
print(json.dumps(personas_json_final[0], ensure_ascii=False, indent=2))

Primer registro CSV modificado:
['animals-african_elephant', 'African Elephant (revisado)', '4']

Primer registro JSON modificado:
{
  "id": "neutral_robert_001",
  "type": "neutral",
  "name": "Robert",
  "profession": "Delivery Person",
  "demographics": {
    "age": "35",
    "region": "Saint Peters, MO, USA"
  },
  "background": "A Delivery Person who watches how friends respond to news and adapts their view accordingly. Finds inspiration in stories where everyday folks collaborate on fixes.",
  "personality_traits": [
    "Switches tone based on how others respond",
    "Worries about missing context and second-guesses self",
    "Looks for human impact whenever policies trend"
  ],
  "communication_style": {
    "tone": "Thoughtful but impressionable",
    "engagement_level": "low"
  },
  "reviewed": true
}


## 4. Generación del resumen TXT

La tercera clase consolida las fuentes, rutas, modificaciones, métodos utilizados y cantidad de registros procesados.

In [8]:
generador = GeneradorResumenTXT()

ruta_resumen = generador.generar_resumen(
    url_csv=URL_CSV,
    url_json=URL_JSON,
    archivo_csv_original=str(RUTA_CSV),
    archivo_json_original=str(RUTA_JSON),
    archivo_csv_modificado=str(SALIDA_CSV),
    archivo_json_modificado=str(SALIDA_JSON),
    cantidad_csv=len(datos_csv),
    cantidad_json=len(personas_json),
    cambio_csv=cambio_csv,
    cambio_json=cambio_json
)

print(ruta_resumen.read_text(encoding="utf-8"))

RESUMEN DEL PROCESAMIENTO DE ARCHIVOS - ACTIVIDAD 2

Fecha de generación: 2026-10-21 03:09:40

1. FUENTES DE DATOS
-------------------
Dataset CSV: https://huggingface.co/datasets/zhugetd/HeightData
Dataset JSON: https://huggingface.co/datasets/loge2146/evocorps-neutral-personas

2. ARCHIVOS ORIGINALES
----------------------
CSV: /workspaces/AD_2026Q4_Grupo20/AE2/datos_originales/heights.csv
JSON: /workspaces/AD_2026Q4_Grupo20/AE2/datos_originales/neutral_personas_database.json

3. MODIFICACIONES REALIZADAS
----------------------------
CSV: Se modificó el nombre del primer registro.
Cambio realizado: African Elephant -> African Elephant (revisado)

JSON: Se agregó el campo 'reviewed' al primer registro.
Cambio realizado: False -> True

4. ARCHIVOS GENERADOS
---------------------
CSV modificado: /workspaces/AD_2026Q4_Grupo20/AE2/datos_modificados/heights_modificado.csv
JSON modificado: /workspaces/AD_2026Q4_Grupo20/AE2/datos_modificados/neutral_personas_database_modificado.json

5. CLAS

## 5. Conclusión

Se implementaron tres clases para leer, modificar y guardar datos en formatos CSV, JSON y TXT. Los cambios se almacenaron en archivos separados, conservando los originales. El informe final documenta las fuentes, las operaciones realizadas y los registros procesados.
Copiar celda 11